# TrainLM T1: TorchTPU single-device probe

This notebook runs device detection and the probe in one Python process. It checks package provenance, forward execution, autograd, an AdamW update, and optionally torch.compile.

Hosted Kaggle currently documents TPU v5e-8, not v5e-1. Colab's public TPU notebook documents TPU v2. Neither hosted accelerator is v5e-1 evidence. For the planned v5e-1 check, use a one-chip Cloud TPU v5e VM (Google lists machine type ct5lp-hightpu-1t) with the approved TorchTPU runtime; a Colab frontend connected to that VM runs on the VM's hardware.

This notebook does not install packages or treat a generic accelerator as TorchTPU. A TPU pass is labeled with the runtime platform and must be paired with the VM/accelerator identifier to establish the hardware generation. Use v5e-8 for checks that require multiple devices.

In [ ]:
from pathlib import Path
import subprocess
import tempfile

PROBE_RELATIVE_PATH = Path("examples/torchtpu_t1_v5e1_probe.py")
PROBE_BRANCH = "docs/01-t1-v5e1-probe"
repo_candidates = [
    Path.cwd(), *Path.cwd().parents,
    Path("/kaggle/working/TrainLM"), Path("/content/TrainLM"),
]
repo_root = next(
    (candidate.resolve() for candidate in repo_candidates
     if (candidate / PROBE_RELATIVE_PATH).is_file()),
    None,
)

# Reuse only a checkout that has the auto-detect probe this notebook expects.
if repo_root is not None:
    probe_source = (repo_root / PROBE_RELATIVE_PATH).read_text()
    if 'default="auto"' not in probe_source:
        repo_root = None

if repo_root is None:
    clone_path = Path(tempfile.mkdtemp(prefix="trainlm-t1-")) / "TrainLM"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", PROBE_BRANCH,
         "https://github.com/Dhiraj309/TrainLM.git", str(clone_path)],
        check=True,
    )
    repo_root = clone_path.resolve()

assert (repo_root / PROBE_RELATIVE_PATH).is_file()
print(f"repository={repo_root}")
print(f"probe_branch={PROBE_BRANCH}")

## Detect runtime and device

The probe uses the official TorchTPU device API and verifies a real tensor allocation inside the probe process. The standard hosted options are documented as Colab TPU v2 and Kaggle v5e-8. A Colab connection to a Cloud TPU VM uses that VM's hardware.

In [ ]:
import os
import sys

if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    NOTEBOOK_HOST = "Colab"
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ:
    NOTEBOOK_HOST = "Kaggle"
else:
    NOTEBOOK_HOST = "other/local runtime"

print(f"notebook_host={NOTEBOOK_HOST}")
DEVICE = "auto"
print(f"selected_device={DEVICE} (resolved inside probe process)")
if NOTEBOOK_HOST == "Kaggle":
    print("hosted_accelerator_note=Kaggle documents TPU v5e-8; this is not v5e-1 evidence")
elif NOTEBOOK_HOST == "Colab":
    print("hosted_accelerator_note=standard Colab TPU is documented as TPU v2; a connected local runtime uses its server hardware")
print("The probe prints whether TPU allocation passed or it selected CPU.")

## Eager single-device smoke

The probe uses a tiny synthetic model and data. It prints Python/PyTorch and installed Torch/TPU/XLA-related distribution versions, then checks finite loss and gradients plus an actual optimizer update. Preserve the complete cell output as evidence.

In [ ]:
import sys
import subprocess

probe = repo_root / PROBE_RELATIVE_PATH

def run_probe(extra_args=()):
    command = [sys.executable, str(probe), "--device", DEVICE, *extra_args]
    print(f"probe_command={command}")
    result = subprocess.run(command, cwd=repo_root, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, file=sys.stderr, end="")
    if result.returncode:
        raise RuntimeError(f"Probe exited with status {result.returncode}; see child output above.")

run_probe()

## Optional compile smoke

Run this separately because compilation can take longer and may expose a runtime/compiler issue. If it fails, keep the error as evidence; the script does not silently fall back to eager execution.

In [ ]:
RUN_COMPILE = False  # Set True to run the optional torch.compile check.
if RUN_COMPILE:
    run_probe(["--compile"])
else:
    print("compile probe skipped; set RUN_COMPILE = True when ready")

## Record results

Record the notebook host, runtime/VM image identifier, date, actual accelerator, selected device, command, Python/PyTorch/distribution versions, and whether import, device placement, forward, autograd, optimizer update, and optional compile passed. The host label identifies the notebook frontend only; keep the accelerator identifier to establish v5e-1 evidence. Do not paste tokens, credentials, or a full environment dump.

This is only a single-device smoke. Device detection and training run in one child process. Move to v5e-8 for collectives, DP8, sharding/FSDP, distributed-checkpoint behavior, or scaling; do not infer those results from this notebook.